In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
import pickle
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from scipy.interpolate import interp1d
import math
from datetime import datetime
from tqdm import tqdm
from sklearn.manifold import TSNE
from umap import UMAP
from deepview.clustering_pytorch.datasets.factory import sliding_window

import plotly.express as px

In [2]:
def AE_eval_time_series(train_loader, model, device):
    model.eval()

    representation_list = []
    sample_list, timestamp_list, label_list, pred_list, timestr_list, flag_list = [], [], [], [], [], []
    for i, (sample, label) in enumerate(train_loader):
        sample = sample.to(device=device, non_blocking=True, dtype=torch.float)

        # input of autoencoder will be 3D, the backbone is 1d-cnn
        x_encoded, output = model(sample)  # x_encoded.shape=batch512,outchannel128,len13
        # print(type(output))
        # x_encoded, output = model(input_).view(b, 2, -1)  # output.shape=b,2,128, split the first dim into 2 parts
        tmp_representation = x_encoded.detach().cpu().numpy()
        representation_list.append(tmp_representation)
        sample_list.append(sample.detach().cpu().numpy())
        label_list.append(label.detach().cpu().numpy())
        pred_list.append(output.detach().cpu().numpy())

    return representation_list, sample_list, pred_list, label_list

def reduce_dimension_with_tsne(array, method='tsne'):
    # tsne or pca
    tsne = TSNE(n_components=2)  # 创建TSNE对象，降维到2维
    reduced_array = tsne.fit_transform(array)  # 对数组进行降维
    return reduced_array

In [3]:
class MSEloss(nn.Module):
    def __init__(self):
        super(MSEloss, self).__init__()

    def forward(self, input, target):
        '''
        input: raw sensor data
        target: reconstructed sensor data
        the mse loss makes the target data to be similar to the input data
        '''
        loss = nn.MSELoss()
        output = loss(input, target)
        return output

In [4]:
class data_loader_umineko(Dataset):
    def __init__(self, samples, labels, device='cpu'):
        self.samples = torch.tensor(samples).to(device)  # check data type
        self.labels = torch.tensor(labels)  # check data type

    def __getitem__(self, index):
        target = self.labels[index]
        sample = self.samples[index]
        return sample, target

    def __len__(self):
        return len(self.labels)

# prepare train loader of accelerometer
with open('data.pkl', 'rb') as f:
    df_ready = pickle.load(f)
sensor_type = 'accel'
batch_size = 512
device = 'cuda'
len_sw = 180
selected_columns = ['acc_x', 'acc_y', 'acc_z',
                   'label_id']  # without timestamps
tmp_b = sliding_window(df_ready[selected_columns], len_sw)

# concatenate list
data_b = tmp_b[:,:,:-1]  # [B, Len, dim-1]
label_b = tmp_b[:,:,-1]  # [B, Len]

train_set_r = data_loader_umineko(data_b, label_b, device=device)
train_loader = DataLoader(train_set_r, batch_size=batch_size,
                           shuffle=False, drop_last=False)

In [5]:
# 每一个文件为一个test loader
test_filenames = df_ready.animal_tag.unique()
test_loaders = []
for f in test_filenames:
    single_df = df_ready[df_ready['animal_tag']==f]
    tmp_b = sliding_window(single_df[selected_columns], len_sw)
    # concatenate list
    data_b = tmp_b[:,:,:-1]  # [B, Len, dim-1]
    label_b = tmp_b[:,:,-1]  # [B, Len]
    test_set_r = data_loader_umineko(data_b, label_b, device=device)
    test_loader = DataLoader(test_set_r, batch_size=batch_size,
                           shuffle=False, drop_last=False)
    test_loaders.append(test_loader)

In [6]:
# Function to calculate mode for each row
def majority_value(arr):
    majority = []
    for row in arr:  
        values, counts = np.unique(row, return_counts=True)
        majority.append(values[np.argmax(counts)])
    return np.array(majority)

In [22]:
label_dict = {
  'ground_stationary': 0,
  'stationary': 0,
  'preening': 0,
  'bathing': 1,
    'bathing_poss': 1,
    'body_shaking': 6,
    'flying_active': 7,
    'flying_passive': 7,
    'foraging': 4,
    'foraging_fish_poss': 4,
    'foraging_insect_poss': 4,
    'foraging_non-fish': 4,
    'foraging_poss': 4,
    'ground_active': 8,
  'flight_take_off': 2,
  'flight_cruising': 3,
  'foraging_dive': 4,
  'surface_seizing': 5,
  'unknown': -1,
  }

# lstm 

In [7]:
from ae_model import Act2Vec

out_channels = 32
device = 'cuda'
model = Act2Vec(out_channels, input_dim=(4503, len_sw, 3))
model = model.to(device)

full_model_path = r'D:\code\DeepView\deepview\calculate_results\AE_GRU_epoch3999_datalen180_accel.pth'

if torch.cuda.is_available():
    model.load_state_dict(torch.load(full_model_path, weights_only=False))
else:
    model.load_state_dict(torch.load(full_model_path, weights_only=False, map_location=torch.device('cpu')))
    

optimizer = torch.optim.Adam(model.parameters(),
                                     weight_decay=0.000001,
                                     lr=0.0001)
criterion = MSEloss()
criterion = criterion.to(device)

In [8]:
representation_list, sample_list, pred_list, label_list = \
            AE_eval_time_series(train_loader, model, device)

# tsne latent representation to shape=(2, len) PCA降维到形状为 (2, len)
repre_concat = np.concatenate(representation_list)
repre_reshape = repre_concat.reshape(repre_concat.shape[0], -1)


In [9]:
label_concat = np.concatenate(label_list)
label_concat_vote = majority_value(label_concat)
label_concat_vote.shape

(4503,)

In [10]:
# 指定color
label_name = np.unique(label_concat_vote)

In [33]:
tsne = TSNE(n_components=2)  # 创建TSNE对象，降维到2维
repre_tsne = tsne.fit_transform(repre_reshape)  # 对数组进行降维

fig = px.scatter(
    repre_tsne, x=0, y=1,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig.show()
# repre_tsne.shape

In [34]:
tsne = TSNE(n_components=3, random_state=0)
projections = tsne.fit_transform(repre_reshape, )

fig = px.scatter_3d(
    projections, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig.update_traces(marker_size=8)
fig.show()

In [15]:
repre_reshape.shape

(4503, 64)

In [36]:
umap_2d = UMAP(n_components=2)
umap_3d = UMAP(n_components=3)

proj_2d = umap_2d.fit_transform(repre_reshape)
proj_3d = umap_3d.fit_transform(repre_reshape)

fig_2d = px.scatter(
    proj_2d, x=0, y=1,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)

fig_3d = px.scatter_3d(
    proj_3d, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)

fig_2d.show()
fig_3d.show()

# cnn ae

In [37]:
from ae_model2 import CNN_AE

model = CNN_AE(3, out_channels)
model = model.to(device)

full_model_path = r'D:\code\DeepView\deepview\calculate_results\cnngru_epoch39999_datalen180_accel.pth'

if torch.cuda.is_available():
    model.load_state_dict(torch.load(full_model_path, weights_only=False))
else:
    model.load_state_dict(torch.load(full_model_path, weights_only=False, map_location=torch.device('cpu')))
    

optimizer = torch.optim.Adam(model.parameters(),
                                     weight_decay=0.000001,
                                     lr=0.0001)
criterion = MSEloss()
criterion = criterion.to(device)

In [38]:
representation_list, sample_list, pred_list, label_list = \
            AE_eval_time_series(train_loader, model, device)

# tsne latent representation to shape=(2, len) PCA降维到形状为 (2, len)
repre_concat = np.concatenate(representation_list)
repre_reshape = repre_concat.reshape(repre_concat.shape[0], -1)
repre_tsne = reduce_dimension_with_tsne(repre_reshape)

label_concat = np.concatenate(label_list)
label_concat_vote = majority_value(label_concat)
label_concat_vote.shape

(4503,)

In [39]:
# 指定color
label_name = np.unique(label_concat_vote)

In [40]:
tsne = TSNE(n_components=2)  # 创建TSNE对象，降维到2维
repre_tsne = tsne.fit_transform(repre_reshape)  # 对数组进行降维

fig = px.scatter(
    repre_tsne, x=0, y=1,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig.show()
# repre_tsne.shape

In [41]:
tsne = TSNE(n_components=3, random_state=0)
projections = tsne.fit_transform(repre_reshape, )

fig = px.scatter_3d(
    projections, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig.update_traces(marker_size=8)
fig.show()

In [42]:
umap_2d = UMAP(n_components=2)
umap_3d = UMAP(n_components=3)

proj_2d = umap_2d.fit_transform(repre_reshape)
proj_3d = umap_3d.fit_transform(repre_reshape)

fig_2d = px.scatter(
    proj_2d, x=0, y=1,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)

fig_3d = px.scatter_3d(
    proj_3d, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)

fig_2d.show()
fig_3d.show()

## dbscan

In [43]:
from sklearn.cluster import DBSCAN, HDBSCAN
# from sklearn.datasets import make_blobs
# fig, axes = plt.subplots(3, 1, figsize=(10, 12))
dbs = DBSCAN(eps=0.3)
data = dbs.fit(proj_3d)

# dbs.labels_

# for idx, scale in enumerate([1, 0.5, 3]):
#     dbs.fit(X * scale)
#     plot(X * scale, dbs.labels_, parameters={"scale": scale, "eps": 0.3}, ax=axes[idx])

array([-1,  0,  0, ...,  8,  8,  8], dtype=int64)

In [44]:
fig_3d = px.scatter_3d(
    proj_3d, x=0, y=1, z=2,
    color=dbs.labels_.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)


fig_3d.show()

## gyro data clustering results

In [45]:

sensor_type = 'gyro'
batch_size = 512
device = 'cuda'
len_sw = 180
selected_columns = ['gyro_x', 'gyro_y', 'gyro_z', 
                   'label_id']  # without timestamps
tmp_b = sliding_window(df_ready[selected_columns], len_sw)

# concatenate list
data_b = tmp_b[:,:,:-1]  # [B, Len, dim-1]
label_b = tmp_b[:,:,-1]  # [B, Len]

train_set_gyro = data_loader_umineko(data_b, label_b, device=device)
train_loader_gyro = DataLoader(train_set_gyro, batch_size=batch_size,
                           shuffle=False, drop_last=False)

In [46]:
# 每一个文件为一个test loader
test_loaders_gyro = []
for f in test_filenames:
    single_df = df_ready[df_ready['animal_tag']==f]
    tmp_b = sliding_window(single_df[selected_columns], len_sw)
    # concatenate list
    data_b = tmp_b[:,:,:-1]  # [B, Len, dim-1]
    label_b = tmp_b[:,:,-1]  # [B, Len]
    test_set_gyro = data_loader_umineko(data_b, label_b, device=device)
    test_loader_gyro = DataLoader(test_set_gyro, batch_size=batch_size,
                           shuffle=False, drop_last=False)
    test_loaders_gyro.append(test_loader_gyro)

In [47]:
representation_list_gyro, sample_list_gyro, pred_list_gyro, label_list_gyro = \
            AE_eval_time_series(train_loader_gyro, model, device)

# tsne latent representation to shape=(2, len) PCA降维到形状为 (2, len)
repre_concat_gyro = np.concatenate(representation_list_gyro)
repre_reshape_gyro = repre_concat_gyro.reshape(repre_concat_gyro.shape[0], -1)


In [48]:
umap_2d = UMAP(n_components=2)
umap_3d = UMAP(n_components=3)

proj_2d_gyro = umap_2d.fit_transform(repre_reshape_gyro)
proj_3d_gyro = umap_3d.fit_transform(repre_reshape_gyro)

fig_2d = px.scatter(
    proj_2d_gyro, x=0, y=1,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)

fig_3d = px.scatter_3d(
    proj_3d_gyro, x=0, y=1, z=2,
    color=label_concat_vote.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)

fig_2d.show()
fig_3d.show()

In [49]:
fig_3d = px.scatter_3d(
    proj_3d_gyro, x=0, y=1, z=2,
    color=dbs.labels_.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)

fig_3d.show()

In [50]:
arr = dbs.labels_
arr[(arr != 1) & (arr != 3)] = 0
fig_3d = px.scatter_3d(
    proj_3d_gyro, x=0, y=1, z=2,
    color=arr.astype(str),  
    labels={'color': 'activity'}
)
fig_3d.update_traces(marker_size=5)

fig_3d.show()